In [1]:
from PINN_utils import *

In [2]:
pde = PDE(lambda x: 0.2, S_injecao=0.9)

In [3]:
N_colocacao = 15000
N_inicial = 1500
N_contorno = 1500
sampler = DomainSampler.create(N_colocacao, N_inicial, N_contorno)

In [4]:
batch_size = 2048
batch_inicial = batch_size // 7
batch_contorno = batch_size // 7
batch_colocacao = batch_size - batch_contorno - batch_inicial

PINN_model = RHPiecewise_PINN(2, 1, 16, 4, t0=0.0, x0=0.0)

n_iter = 10000
lr = 1e-2
optimizer = optim.Adam(itertools.chain(PINN_model.net_IC.parameters(), PINN_model.net_BC.parameters()), lr=lr)
optimizer_RH = optim.Adam(PINN_model.net_RH.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_iter, eta_min=1e-6)

# a cada quantas iterações vamos alterar os pontos usados
N_change_points = 75

ts_treino = torch.linspace(0.0, 1.0, 500).reshape(-1, 1).requires_grad_(True)

In [5]:
for it in range(n_iter):
    
    batch = sampler.sample_batch(batch_colocacao, batch_inicial, batch_contorno)
    descontinuidade = PINN_model.get_discontinuity(ts_treino)
    mask = ((descontinuidade > sampler.xmin) & (descontinuidade < sampler.xmax)).flatten()
    y_fake = PINN_model(batch.get_all())

    L_f = pde.pde_loss(batch.get_colocacao(), y_fake[batch.slice_colocacao])
    L_b1 = pde.IC_loss(batch.get_inicial(), y_fake[batch.slice_inicial])
    L_b2 = pde.BC_loss(batch.get_contorno(), y_fake[batch.slice_contorno])
    L_OL = pde.Welge_loss(torch.column_stack((ts_treino[mask], descontinuidade[mask])), PINN_model.net_IC, PINN_model.net_BC)
    
    L_PINN = L_f + 10 * (L_b1 + L_b2) + it / n_iter * L_OL
    
    optimizer.zero_grad()
    L_PINN.backward()
    optimizer.step()
    scheduler.step()
    
    L_RH = pde.RH_loss(ts_treino, descontinuidade, mask, PINN_model.net_IC, PINN_model.net_BC)
    
    optimizer_RH.zero_grad()
    L_RH.backward()
    optimizer_RH.step()

    if it % N_change_points == 0:
        sampler.resample(PINN_model, pde)
    
    if (it + 1) % (n_iter // 10) == 0:
        print(f"Iter {it+1}: L_PINN={L_PINN.item():.10f}")

Iter 1000: L_PINN=0.0069814133
Iter 2000: L_PINN=0.0086608110
Iter 3000: L_PINN=0.0080721043
Iter 4000: L_PINN=0.0116212219
Iter 5000: L_PINN=0.0164725706
Iter 6000: L_PINN=0.0063346336
Iter 7000: L_PINN=0.0084837563
Iter 8000: L_PINN=0.0054166075
Iter 9000: L_PINN=0.0130809564
Iter 10000: L_PINN=0.0186622366


In [6]:
PINN_model.save("resultados/prob2.pth")